# Biomapper2 API Tutorial: Mapping Metabolites to Standardized Identifiers

**Audience**: Phenome Health team members new to the Biomapper2 API  
**Prerequisites**: Basic Python, Pandas, async/await concepts  
**Time**: ~20 minutes to complete

## Learning Objectives

By the end of this tutorial, you will be able to:

1. **Understand** how Biomapper2 maps entity names to standardized knowledge graph identifiers
2. **Configure** API requests with entity types, identifier hints, and annotation modes
3. **Preprocess** compound names for better API resolution (quote stripping, suffix removal)
4. **Interpret** API responses including confidence scores and assigned identifiers
5. **Execute** batch mapping with rate limiting and progress tracking
6. **Analyze** results to assess mapping quality

## Real-World Context

This tutorial is grounded in a real mapping project:

| Metric | Value |
|--------|-------|
| **Dataset** | Metabolon unknown combined features |
| **Total features** | 2,221 |
| **Features mapped** | 1,804 (81.2%) |
| **Resolution rate** | 96.1% (1,218/1,267 unique names) |
| **With HMDB hints** | 100% resolution |
| **Without hints** | 93.6% resolution |
| **Net new HMDB annotations** | 65 compounds |
| **New vocabularies added** | PubChem (188), CHEBI (301), RefMet (342) |

## Prerequisites

Before running this notebook, set the `BIOMAPPER_API_KEY` environment variable:

```bash
export BIOMAPPER_API_KEY=your-api-key-here
```

Or add it to a `.env` file in the project root.

## Key Concepts Preview

Before diving in, let's preview the key concepts you'll encounter:

| Concept | Description |
|---------|-------------|
| **biolink:SmallMolecule** | Entity type for metabolites. The Biolink Model standardizes entity types across knowledge graphs, enabling automatic routing to appropriate annotators. |
| **annotation_mode** | Controls when the API queries external databases. `"missing"` (default) only annotates entities without provided identifiers. |
| **identifier hints** | Pre-existing IDs (e.g., HMDB) that help the resolver pick the correct candidate when multiple matches exist. |
| **confidence scores** | Numeric scores indicating match quality. Higher = better (≥2.0 is high confidence). |
| **curies** | Compact URIs like `CHEBI:15971` that uniquely identify entities across knowledge graphs. |
| **assigned_ids** | All identifiers found, organized by annotator (metabolomics-workbench, kestrel-hybrid-search) and vocabulary. |

---
## Part 1: Setup & API Understanding

### Setup & Configuration

We start by importing required libraries and configuring the API connection. Key configuration includes:
- **API key**: Required for authentication
- **Rate limiting**: Prevents overwhelming the API (0.3s delay between calls)
- **Output paths**: Where results will be saved

In [1]:
import sys
import os
import asyncio
import json
import re
from pathlib import Path
from typing import Any

import pandas as pd
import httpx
from dotenv import load_dotenv
from tqdm.auto import tqdm

# Handle Jupyter's event loop (required for async code in notebooks)
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

# Load environment variables from .env file
PROJECT_ROOT = Path("__file__").resolve().parents[2] if "__file__" in dir() else Path.cwd().parents[1]
load_dotenv(PROJECT_ROOT / ".env")

# Verify API key is available
BIOMAPPER_API_KEY = os.getenv("BIOMAPPER_API_KEY")
if not BIOMAPPER_API_KEY:
    raise EnvironmentError(
        "BIOMAPPER_API_KEY not found in environment.\n"
        "Set it before running: export BIOMAPPER_API_KEY=your-key-here\n"
        "Or add to .env file in project root."
    )
print(f"✓ BIOMAPPER_API_KEY configured (length: {len(BIOMAPPER_API_KEY)})")

# Biomapper API configuration
BIOMAPPER_BASE_URL = "https://biomapper.expertintheloop.io/api/v1"

# Rate limiting to avoid overwhelming the API
# Recommended: 0.3s between calls for batch processing
RATE_LIMIT_DELAY = 0.3  # seconds between API calls

# Optional: Limit number of compounds to process (set to None for all)
# TIP: Start with a small LIMIT (e.g., 10) when testing your workflow
LIMIT = None  # Change to e.g. 10 for testing

# File paths - customize these for your data
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1] if "notebooks" in str(NOTEBOOK_DIR) else NOTEBOOK_DIR

# TODO: Update these paths for your data
INPUT_FILE = PROJECT_ROOT / "data" / "metabolon" / "raw" / "Metabolon_unknown_combined_features_metadata.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "data" / "metabolon" / "processed"
REVIEW_DIR = PROJECT_ROOT / "data" / "review"

OUTPUT_JSON = OUTPUT_DIR / "tutorial_biomapper_mapping.json"
OUTPUT_TSV = REVIEW_DIR / "tutorial_biomapper_mapping.tsv"

# Ensure output directories exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input file: {INPUT_FILE} (exists: {INPUT_FILE.exists()})")
print(f"Output JSON: {OUTPUT_JSON}")
print(f"Output TSV: {OUTPUT_TSV}")

✓ BIOMAPPER_API_KEY configured (length: 43)
Input file: /home/trentleslie/Insync/projects/biovector-eval/data/metabolon/raw/Metabolon_unknown_combined_features_metadata.xlsx (exists: True)
Output JSON: /home/trentleslie/Insync/projects/biovector-eval/data/metabolon/processed/tutorial_biomapper_mapping.json
Output TSV: /home/trentleslie/Insync/projects/biovector-eval/data/review/tutorial_biomapper_mapping.tsv


### Health Check

**Always verify API connectivity before processing data.** This prevents wasted time if credentials are incorrect or the service is unavailable.

In [2]:
async def check_api_health():
    """Verify Biomapper2 API connectivity."""
    async with httpx.AsyncClient() as client:
        response = await client.get(
            f"{BIOMAPPER_BASE_URL}/health",
            headers={"X-API-Key": BIOMAPPER_API_KEY},
            timeout=10.0,
        )
        response.raise_for_status()
        return response.json()

# Run health check
health = asyncio.get_event_loop().run_until_complete(check_api_health())
print(f"✓ API Health: {health}")

# Validate key fields
if health.get("status") == "healthy" and health.get("mapper_initialized"):
    print("✓ API is ready to process requests")
else:
    print("⚠️ API may not be fully operational - check response above")

✓ API Health: {'status': 'healthy', 'version': '0.1.0', 'mapper_initialized': True}
✓ API is ready to process requests


### Smoke Test with L-Histidine

Before building your full pipeline, **test with a known compound** to understand the response schema. L-Histidine is a well-characterized amino acid that should resolve cleanly.

Pay attention to the response structure:
- **curies**: List of standardized identifiers found
- **chosen_kg_id**: Primary identifier selected by the resolver
- **assigned_ids**: Detailed breakdown by annotator and vocabulary

In [3]:
async def smoke_test_api():
    """Test API with a known compound to verify schema."""
    async with httpx.AsyncClient() as client:
        # Test with known compound: L-Histidine
        payload = {
            "name": "L-Histidine",
            "entity_type": "biolink:SmallMolecule",
            "options": {"annotation_mode": "missing"},
        }
        
        response = await client.post(
            f"{BIOMAPPER_BASE_URL}/map/entity",
            json=payload,
            headers={"X-API-Key": BIOMAPPER_API_KEY},
            timeout=30.0,
        )
        response.raise_for_status()
        return response.json()

# Run smoke test
smoke_result = asyncio.get_event_loop().run_until_complete(smoke_test_api())

print("=" * 60)
print("FULL RAW API RESPONSE:")
print("=" * 60)
print(json.dumps(smoke_result, indent=2))
print()

# ============================================================
# RESPONSE INTERPRETATION GUIDE
# ============================================================
# Let's break down what each field means:

if "result" in smoke_result:
    r = smoke_result["result"]
    print("=" * 60)
    print("RESPONSE INTERPRETATION:")
    print("=" * 60)
    
    # 1. name - Echo of input (or normalized version)
    print(f"\n1. name: '{r.get('name')}'")
    print("   → This is your input compound name")
    
    # 2. curies - List of standardized identifiers
    curies = r.get('curies', [])
    print(f"\n2. curies: {curies}")
    print("   → CURIEs (Compact URIs) are standardized identifiers")
    print("   → Format: PREFIX:ID (e.g., CHEBI:15971)")
    if curies:
        print(f"   → First CURIE ({curies[0]}) is the primary result")
    
    # 3. chosen_kg_id - The resolver's selected knowledge graph ID
    print(f"\n3. chosen_kg_id: {r.get('chosen_kg_id')}")
    print("   → When multiple candidates exist, the resolver votes to select one")
    
    # 4. assigned_ids - Breakdown by annotator
    assigned_ids = r.get('assigned_ids', {})
    print(f"\n4. assigned_ids structure: {list(assigned_ids.keys())}")
    print("   → metabolomics-workbench: Queries RefMet database (metabolite names)")
    print("   → kestrel-hybrid-search: Queries KRAKEN knowledge graph (semantic search)")
    
    # Extract and explain confidence score
    for annotator, vocabs in assigned_ids.items():
        for vocab, codes in vocabs.items():
            for code, meta in codes.items():
                if isinstance(meta, dict) and "score" in meta:
                    score = meta["score"]
                    print(f"\n5. Confidence score: {score:.3f}")
                    if score >= 2.0:
                        print("   → HIGH confidence (≥2.0) - strong match")
                    elif score >= 1.0:
                        print("   → MEDIUM confidence (1.0-2.0) - likely correct")
                    else:
                        print("   → LOWER confidence (<1.0) - may need review")
                    break

print("\n" + "=" * 60)
print("✓ Smoke test passed - API is working correctly")

FULL RAW API RESPONSE:
{
  "result": {
    "name": "L-Histidine",
    "curies": [
      "RM:0129894",
      "CHEBI:15971"
    ],
    "chosen_kg_id": "CHEBI:15971",
    "kg_ids": {
      "CHEBI:15971": [
        "RM:0129894",
        "CHEBI:15971"
      ]
    },
    "assigned_ids": {
      "metabolomics-workbench": {
        "refmet_id": {
          "RM0129894": {}
        }
      },
      "kestrel-hybrid-search": {
        "CHEBI": {
          "15971": {
            "score": 2.4898567923359374
          }
        }
      }
    },
    "error": null
  },
  "metadata": {
    "request_id": "db11c3a5-f88d-4030-a91f-f41937050767",
    "processing_time_ms": 12.68
  }
}

RESPONSE INTERPRETATION:

1. name: 'L-Histidine'
   → This is your input compound name

2. curies: ['RM:0129894', 'CHEBI:15971']
   → CURIEs (Compact URIs) are standardized identifiers
   → Format: PREFIX:ID (e.g., CHEBI:15971)
   → First CURIE (RM:0129894) is the primary result

3. chosen_kg_id: CHEBI:15971
   → When multiple

---
## Part 2: Data Preparation

### Load Sample Data

Load your metabolomics data and explore its structure. The key columns are:
- **matched_name**: The compound name to query
- **ms1_compound_name**: May contain HMDB identifiers as hints
- **match_level**: Quality tier (MS1 > MS2 > CURATION)

In [4]:
# Load the data
# TODO: Replace with your data loading code
df = pd.read_excel(INPUT_FILE)

print(f"Loaded {len(df)} features")
print(f"Columns: {list(df.columns)}")
print()

# Completeness statistics - understand your data quality
print("=" * 60)
print("DATA COMPLETENESS")
print("=" * 60)

# Check for key columns (adjust these for your data)
key_cols = ['matched_name', 'ms1_compound_name', 'ms2_compound_name', 'match_level']
for col in key_cols:
    if col in df.columns:
        count = df[col].notna().sum()
        pct = count / len(df) * 100
        print(f"  {col}: {count}/{len(df)} ({pct:.1f}%)")
    else:
        print(f"  {col}: NOT FOUND in dataset")

print()

# Show distribution of match levels if available
if 'match_level' in df.columns:
    print("=" * 60)
    print("MATCH LEVEL DISTRIBUTION")
    print("=" * 60)
    print(df['match_level'].value_counts().to_string())

# Preview data
print()
print("=" * 60)
print("SAMPLE DATA")
print("=" * 60)
df.head(3)

Loaded 2221 features
Columns: ['feature_id', 'mean_mz', 'mean_ri', 'n_peaks', 'group_id', 'is_base_peak', 'adduct_type', 'isotope_annotation', 'neutral_mass', 'adduct_group_id', 'ms1_spectrum_id', 'ms1_cosine_score', 'ms1_isotopes_matched', 'ms1_compound_name', 'ms2_spectrum_id', 'ms2_score', 'ms2_n_matches', 'ms2_compound_name', 'matched_comp_id', 'group_ms2_source_feature_id', 'group_ms2_spectrum_id', 'group_ms2_compound_name', 'group_ms2_score', 'group_ms2_n_matches', 'curation_chemical_id', 'curation_compound_name', 'curation_score', 'matched_name', 'match_level', 'pathway_map', 'pathway_probability']

DATA COMPLETENESS
  matched_name: 1804/2221 (81.2%)
  ms1_compound_name: 857/2221 (38.6%)
  ms2_compound_name: 1275/2221 (57.4%)
  match_level: 2221/2221 (100.0%)

MATCH LEVEL DISTRIBUTION
match_level
MS2         890
CURATION    828
MS1         503

SAMPLE DATA


,feature_id,mean_mz,mean_ri,n_peaks,group_id,is_base_peak,adduct_type,isotope_annotation,neutral_mass,adduct_group_id,...,group_ms2_compound_name,group_ms2_score,group_ms2_n_matches,curation_chemical_id,curation_compound_name,curation_score,matched_name,match_level,pathway_map,pathway_probability
0,method_1_29905,212.118167,3253,67,546,True,[M+H]+,M0,211.111439,3096,...,"""1,3-Diphenylguanidine_CE45""",0.994698,8.0,NaN,NaN,NaN,"""1,3-Diphenylguanidine_CE45""",MS2,NaN,NaN
1,method_1_1434,85.028349,661,61,105,True,[M+H]+,M0,84.021621,777,...,"""3-Amino-1,2,4-triazol""",0.968043,2.0,NaN,NaN,NaN,"""3-Amino-1,2,4-triazol""",MS2,NaN,NaN
2,method_1_17931,159.016457,601,61,499,True,[M+H]+,M0,158.009729,658,...,"""4,6-DIOXOHEPTANOIC ACID""",0.874336,3.0,NaN,NaN,NaN,"""4,6-DIOXOHEPTANOIC ACID""",MS2,NaN,NaN


### Name Preprocessing

**Why preprocessing matters**: Raw compound names often contain artifacts that hurt API resolution:
- Extraneous quotes (`"Glucose"`)
- Whitespace issues
- Vendor-specific suffixes (`_CE45` from collision energy annotations)

**Important caveat**: In our Metabolon investigation, only **2 of 49 unresolved compounds** failed due to `_CE` suffixes. This preprocessing is a best practice, but most unresolved compounds fail due to vendor codes (Z-codes, AKOS) or knowledge graph gaps, not formatting issues.

In [5]:
def clean_compound_name(name: str) -> str | None:
    """Clean compound name for better API resolution.
    
    Steps:
    1. Handle NA/empty values
    2. Strip leading/trailing quotes
    3. Strip _CE## suffixes (collision energy annotations)
    4. Clean up whitespace
    """
    if pd.isna(name):
        return None
    
    name = str(name).strip()
    
    # Strip leading/trailing quotes
    if name.startswith('"') and name.endswith('"'):
        name = name[1:-1]
    
    # Strip _CE## suffixes (collision energy annotations from MS data)
    # Pattern matches _CE followed by digits at end of string
    name = re.sub(r'_CE\d+$', '', name)
    
    return name.strip() if name else None


# Demonstrate preprocessing with examples
print("=" * 60)
print("PREPROCESSING EXAMPLES")
print("=" * 60)

examples = [
    '"1,3-Diphenylguanidine_CE45"',  # Quoted + CE suffix
    'L-Histidine',                    # Already clean
    '  Glucose  ',                    # Whitespace
    '"3-Amino-1,2,4-triazol"',        # Just quoted
]

for raw in examples:
    cleaned = clean_compound_name(raw)
    if raw != cleaned:
        print(f"  {repr(raw):45} → {repr(cleaned)}")
    else:
        print(f"  {repr(raw):45} (unchanged)")

print()
print("⚠️ NOTE: CE suffix removal helps some compounds resolve, but")
print("   most failures are due to vendor codes or knowledge gaps.")

PREPROCESSING EXAMPLES
  '"1,3-Diphenylguanidine_CE45"'                → '1,3-Diphenylguanidine'
  'L-Histidine'                                 (unchanged)
  '  Glucose  '                                 → 'Glucose'
  '"3-Amino-1,2,4-triazol"'                     → '3-Amino-1,2,4-triazol'

⚠️ NOTE: CE suffix removal helps some compounds resolve, but
   most failures are due to vendor codes or knowledge gaps.


### Extract Identifier Hints

**Why hints matter**: Providing existing identifiers (like HMDB IDs) dramatically improves resolution:
- **With HMDB hints**: 100% resolution rate
- **Without hints**: 93.6% resolution rate

**Common misconception**: "I provided an HMDB ID, why didn't it just use that?"  
**Answer**: Hints inform the resolver, not the annotators. The API still runs its annotators (metabolomics-workbench, kestrel-hybrid-search) to find candidates. Your hint helps the resolver pick the correct candidate when there are multiple matches, but it doesn't trigger a direct lookup.

In [6]:
def extract_hmdb_id(ms1_name: str) -> str | None:
    """Extract HMDB identifier from ms1_compound_name format.
    
    Common formats:
    - 'HMDB:HMDBXXXXX-YYYY CompoundName'
    - 'HMDB00177'
    
    Returns: 'HMDBXXXXX' (just the ID part)
    """
    if pd.isna(ms1_name):
        return None
    match = re.search(r'HMDB:(HMDB\d+)', str(ms1_name))
    if match:
        return match.group(1)
    # Fallback: bare HMDB ID
    match = re.search(r'(HMDB\d{5,7})', str(ms1_name))
    return match.group(1) if match else None


# Show extraction examples
print("=" * 60)
print("HMDB EXTRACTION EXAMPLES")
print("=" * 60)

if 'ms1_compound_name' in df.columns:
    ms1_names = df['ms1_compound_name'].dropna().astype(str)
    hmdb_samples = ms1_names[ms1_names.str.contains('HMDB', na=False)].head(5)
    
    for name in hmdb_samples:
        hmdb_id = extract_hmdb_id(name)
        print(f"  {name[:50]:50} → {hmdb_id}")
    
    # Statistics
    total_with_ms1 = len(ms1_names)
    total_with_hmdb = ms1_names.apply(extract_hmdb_id).notna().sum()
    print()
    print(f"✓ Features with extractable HMDB hints: {total_with_hmdb}/{total_with_ms1}")
else:
    print("⚠️ No 'ms1_compound_name' column found - adjust extraction for your data format")

HMDB EXTRACTION EXAMPLES
  HMDB:HMDB03349-2257 L-Dihydroorotic acid           → HMDB03349
  HMDB:HMDB03320-2242 Indole-3-carboxylic acid       → HMDB03320
  HMDB:HMDB00635-869 Succinylacetone                 → HMDB00635
  HMDB:HMDB00174-274 L-Fucose                        → HMDB00174
  HMDB:HMDB00177-284 L-Histidine                     → HMDB00177

✓ Features with extractable HMDB hints: 857/857


### Build Mapping Queue

**Deduplication strategy**: The same compound name may appear multiple times in your dataset. We:
1. Deduplicate by cleaned name
2. Keep the first HMDB hint per name (if multiple exist)

In [7]:
# Build mapping records from data
# TODO: Update these column names for your data
NAME_COLUMN = 'matched_name'  # Column containing compound names
HINT_COLUMN = 'ms1_compound_name'  # Column containing HMDB hints (optional)

mapping_records = []

for idx, row in df.iterrows():
    matched_name = clean_compound_name(row.get(NAME_COLUMN))
    if not matched_name:
        continue  # Skip rows without compound names
    
    hmdb_hint = None
    if HINT_COLUMN in df.columns:
        hmdb_hint = extract_hmdb_id(row.get(HINT_COLUMN))
    
    mapping_records.append({
        'feature_id': row.get('feature_id', idx),
        'matched_name': matched_name,
        'match_level': row.get('match_level'),
        'hmdb_hint': hmdb_hint,
    })

print(f"Features with compound names: {len(mapping_records)}")

# Deduplicate by name, keeping first hint per name
unique_names = {}
for rec in mapping_records:
    name = rec['matched_name']
    if name not in unique_names:
        unique_names[name] = rec['hmdb_hint']

print(f"Unique compound names: {len(unique_names)}")

# Analyze hint availability
with_hints = sum(1 for h in unique_names.values() if h is not None)
without_hints = len(unique_names) - with_hints
print(f"  With HMDB hints: {with_hints}")
print(f"  Without hints: {without_hints}")

# Create mapping queue
mapping_queue = list(unique_names.items())

# Apply limit if set (for testing)
if LIMIT is not None:
    mapping_queue = mapping_queue[:LIMIT]
    print(f"\n⚠️ Limited to first {LIMIT} names for testing")

print(f"\n✓ Mapping queue size: {len(mapping_queue)}")

Features with compound names: 1804
Unique compound names: 1267
  With HMDB hints: 502
  Without hints: 765

✓ Mapping queue size: 1267


---
## Part 3: API Execution

### Understanding the Request Structure

Before implementing the client, let's understand what each request field means:

```python
payload = {
    "name": "L-Histidine",              # Compound name to map
    "entity_type": "biolink:SmallMolecule",  # Entity type for routing
    "identifiers": {"HMDB": "HMDB00177"},    # Optional hints
    "options": {"annotation_mode": "missing"},  # Control annotation behavior
}
```

**entity_type**: For metabolites, use `biolink:SmallMolecule`. This automatically selects:
- **metabolomics-workbench**: Queries RefMet database
- **kestrel-hybrid-search**: Queries KRAKEN knowledge graph with semantic search

**identifiers**: Optional dictionary of known IDs. Format: `{"VOCABULARY": "id_value"}`
- These are **hints**, not direct lookups
- Help the resolver choose among multiple candidates

> **Common misconception**: "I provided an HMDB ID, why didn't it just use that?" The answer is that hints inform resolution, not annotation. The API still runs its annotators — the hint helps the resolver pick the correct candidate when multiple matches exist.

**annotation_mode**: 
- `"missing"` (default): Only annotate entities that don't have identifiers in the request
- `"all"`: Always run annotation regardless of provided identifiers
- `"none"`: Skip annotation entirely

For fresh annotation with no pre-existing IDs, `"missing"` and `"all"` behave identically.

### API Client Implementation

Key implementation details:
- **Async client**: Enables rate limiting between calls
- **Error handling**: Graceful degradation for failed requests
- **Result extraction**: Normalize nested response structure

In [8]:
async def map_compound_biomapper(
    client: httpx.AsyncClient,
    name: str,
    hmdb_hint: str | None = None,
) -> dict[str, Any]:
    """Map a compound using the Biomapper API.
    
    Args:
        client: httpx async client
        name: Compound name to map
        hmdb_hint: Optional HMDB identifier hint
    
    Returns:
        API response dict or error dict
    """
    payload = {
        "name": name,
        "entity_type": "biolink:SmallMolecule",
        "options": {"annotation_mode": "missing"},
    }
    
    if hmdb_hint:
        payload["identifiers"] = {"HMDB": hmdb_hint}
    
    try:
        response = await client.post(
            f"{BIOMAPPER_BASE_URL}/map/entity",
            json=payload,
            headers={"X-API-Key": BIOMAPPER_API_KEY},
            timeout=30.0,
        )
        response.raise_for_status()
        return response.json()
    except httpx.HTTPStatusError as e:
        return {"error": f"HTTP {e.response.status_code}: {e.response.text}"}
    except Exception as e:
        return {"error": str(e)}


def extract_result_fields(response: dict, name: str, hmdb_hint: str | None) -> dict:
    """Extract key fields from Biomapper API response into a flat structure."""
    result = {
        "query_name": name,
        "hmdb_hint": hmdb_hint,
        "resolved": False,
        "biomapper_curie": None,
        "biomapper_name": None,
        "biomapper_kg_id": None,
        "confidence_score": None,
        "identifiers": {},
        "error": None,
    }
    
    if "error" in response:
        result["error"] = response["error"]
        return result
    
    if "result" not in response or not response["result"]:
        result["error"] = "Empty result"
        return result
    
    r = response["result"]
    
    # Extract main identifiers
    curies = r.get("curies", [])
    if curies:
        result["resolved"] = True
        result["biomapper_curie"] = curies[0]
    
    result["biomapper_name"] = r.get("name")
    result["biomapper_kg_id"] = r.get("chosen_kg_id")
    
    # Extract confidence score and identifiers from assigned_ids
    assigned_ids = r.get("assigned_ids", {})
    for annotator, vocabs in assigned_ids.items():
        for vocab, codes in vocabs.items():
            for code, meta in codes.items():
                if isinstance(meta, dict):
                    if "score" in meta and result["confidence_score"] is None:
                        result["confidence_score"] = meta["score"]
                    # Collect identifiers by vocabulary
                    if vocab not in result["identifiers"]:
                        result["identifiers"][vocab] = []
                    result["identifiers"][vocab].append(code)
    
    return result


print("✓ API client functions defined")

✓ API client functions defined


### Execute Mapping

Now we run the full mapping with progress tracking. **This will take time** for large datasets:
- 1,000 compounds ≈ 5-10 minutes with 0.3s rate limiting
- Consider starting with `LIMIT=50` to validate your workflow

In [9]:
async def run_mapping(mapping_queue: list[tuple[str, str | None]]) -> list[dict]:
    """Run the full mapping process for all compounds."""
    results = []
    
    async with httpx.AsyncClient() as client:
        print(f"Starting mapping of {len(mapping_queue)} unique compound names...")
        print(f"Estimated time: {len(mapping_queue) * RATE_LIMIT_DELAY / 60:.1f} minutes")
        print()
        
        for name, hmdb_hint in tqdm(mapping_queue, desc="Mapping compounds"):
            # Rate limiting
            await asyncio.sleep(RATE_LIMIT_DELAY)
            
            response = await map_compound_biomapper(client, name, hmdb_hint)
            result = extract_result_fields(response, name, hmdb_hint)
            results.append(result)
    
    return results


# Run the mapping
api_results = asyncio.get_event_loop().run_until_complete(run_mapping(mapping_queue))
print()
print(f"✓ Mapping complete: {len(api_results)} compounds processed")

Starting mapping of 1267 unique compound names...
Estimated time: 6.3 minutes



Mapping compounds:   0%|          | 0/1267 [00:00<?, ?it/s]


✓ Mapping complete: 1267 compounds processed


---
## Part 4: Results Analysis

### Resolution Summary

Calculate overall success rates and compare by hint availability.

In [10]:
# Calculate resolution statistics
resolved = [r for r in api_results if r["resolved"]]
unresolved = [r for r in api_results if not r["resolved"]]
errors = [r for r in api_results if r["error"]]

print("=" * 60)
print("RESOLUTION SUMMARY")
print("=" * 60)
print(f"Total unique names queried: {len(api_results)}")
print(f"Resolved: {len(resolved)} ({len(resolved)/len(api_results)*100:.1f}%)")
print(f"Unresolved: {len(unresolved)} ({len(unresolved)/len(api_results)*100:.1f}%)")
print(f"Errors: {len(errors)} ({len(errors)/len(api_results)*100:.1f}%)")
print()

# Resolution by HMDB hint availability
print("=" * 60)
print("RESOLUTION BY HINT AVAILABILITY")
print("=" * 60)
with_hint = [r for r in api_results if r["hmdb_hint"]]
without_hint = [r for r in api_results if not r["hmdb_hint"]]

if with_hint:
    with_hint_resolved = sum(1 for r in with_hint if r["resolved"])
    print(f"With HMDB hint: {with_hint_resolved}/{len(with_hint)} resolved ({with_hint_resolved/len(with_hint)*100:.1f}%)")
else:
    print("No features with HMDB hints")

if without_hint:
    without_hint_resolved = sum(1 for r in without_hint if r["resolved"])
    print(f"Without hint: {without_hint_resolved}/{len(without_hint)} resolved ({without_hint_resolved/len(without_hint)*100:.1f}%)")
else:
    print("All features have HMDB hints")

# Show unresolved examples
if unresolved:
    print()
    print("=" * 60)
    print(f"SAMPLE UNRESOLVED COMPOUNDS ({min(10, len(unresolved))} of {len(unresolved)})")
    print("=" * 60)
    for r in unresolved[:10]:
        print(f"  {r['query_name'][:50]:50} (hint: {r['hmdb_hint'] or 'none'})")

RESOLUTION SUMMARY
Total unique names queried: 1267
Resolved: 1220 (96.3%)
Unresolved: 47 (3.7%)
Errors: 0 (0.0%)

RESOLUTION BY HINT AVAILABILITY
With HMDB hint: 502/502 resolved (100.0%)
Without hint: 718/765 resolved (93.9%)

SAMPLE UNRESOLVED COMPOUNDS (10 of 47)
  EiM07-29661                                        (hint: none)
  putrescine-C10:0                                   (hint: none)
  Z1005800534                                        (hint: none)
  AKOS016382737                                      (hint: none)
  AKOS034142159                                      (hint: none)
  GBB-200061 [CCS=220.68775939941406]                (hint: none)
  MEGxp0_000907                                      (hint: none)
  MSQ4Y6W7VA [CCS=240.8450927734375]                 (hint: none)
  SCHEMBL9924588                                     (hint: none)
  ST007035 CollisionEnergy:205060                    (hint: none)


### Confidence Distribution

Confidence scores help prioritize which mappings need manual review.

| Score Range | Interpretation | Action |
|-------------|----------------|--------|
| ≥ 2.0 | High confidence | Accept without review |
| 1.0 - 2.0 | Medium confidence | Quick check recommended |
| < 1.0 | Lower confidence | Manual review recommended |

In [11]:
# Confidence score distribution
scores = [r["confidence_score"] for r in resolved if r["confidence_score"] is not None]

if scores:
    print("=" * 60)
    print("CONFIDENCE SCORE DISTRIBUTION")
    print("=" * 60)
    print(f"Min: {min(scores):.3f}")
    print(f"Max: {max(scores):.3f}")
    print(f"Mean: {sum(scores)/len(scores):.3f}")
    print(f"Median: {sorted(scores)[len(scores)//2]:.3f}")
    print()
    
    # Score buckets
    high_conf = sum(1 for s in scores if s >= 2.0)
    med_conf = sum(1 for s in scores if 1.0 <= s < 2.0)
    low_conf = sum(1 for s in scores if s < 1.0)
    
    total = len(scores)
    print(f"High confidence (≥2.0): {high_conf:5} ({high_conf/total*100:5.1f}%) - Accept without review")
    print(f"Medium (1.0-2.0):       {med_conf:5} ({med_conf/total*100:5.1f}%) - Quick check recommended")
    print(f"Lower (<1.0):           {low_conf:5} ({low_conf/total*100:5.1f}%) - Manual review recommended")
else:
    print("No confidence scores available")

CONFIDENCE SCORE DISTRIBUTION
Min: 0.521
Max: 4.955
Mean: 1.759
Median: 0.888

High confidence (≥2.0):   305 ( 42.5%) - Accept without review
Medium (1.0-2.0):          33 (  4.6%) - Quick check recommended
Lower (<1.0):             379 ( 52.9%) - Manual review recommended


### Identifier Vocabulary Coverage

See which identifier types were assigned. Different vocabularies are useful for different downstream applications:
- **refmet_id**: Metabolomics Workbench reference (good for MS data)
- **CHEBI**: Chemical entities of biological interest (structural queries)
- **PUBCHEM**: PubChem compound IDs (extensive chemical property data)

In [12]:
# Count identifiers by vocabulary
vocab_counts = {}
for r in resolved:
    for vocab in r["identifiers"].keys():
        vocab_counts[vocab] = vocab_counts.get(vocab, 0) + 1

print("=" * 60)
print("IDENTIFIER VOCABULARY COVERAGE")
print("=" * 60)

if vocab_counts:
    for vocab, count in sorted(vocab_counts.items(), key=lambda x: -x[1])[:15]:
        print(f"  {vocab:25}: {count:5} ({count/len(resolved)*100:5.1f}%)")
else:
    print("No vocabulary data available")

# Priority identifiers for metabolomics
print()
print("=" * 60)
print("PRIORITY IDENTIFIERS FOR METABOLOMICS")
print("=" * 60)

priority_vocabs = [
    ("refmet_id", "RefMet (Metabolomics Workbench)"),
    ("CHEBI", "CHEBI (Chemical Entities)"),
    ("PUBCHEM.COMPOUND", "PubChem Compound"),
    ("HMDB", "HMDB (Human Metabolome)"),
]

for vocab_key, description in priority_vocabs:
    count = vocab_counts.get(vocab_key, 0)
    print(f"  {description:35}: {count:5} ({count/len(resolved)*100 if resolved else 0:5.1f}%)")

IDENTIFIER VOCABULARY COVERAGE
  refmet_id                :   279 ( 22.9%)
  CHEBI                    :   250 ( 20.5%)
  PUBCHEM.COMPOUND         :   153 ( 12.5%)
  UMLS                     :    62 (  5.1%)
  UNII                     :    41 (  3.4%)
  EFO                      :    36 (  3.0%)
  MESH                     :    28 (  2.3%)
  CHEMBL.COMPOUND          :    22 (  1.8%)
  HP                       :    16 (  1.3%)
  NCBIGene                 :    16 (  1.3%)
  RM                       :    14 (  1.1%)
  USZIPCODE                :    14 (  1.1%)
  NCIT                     :    13 (  1.1%)
  ttd.target               :    11 (  0.9%)
  NCBITaxon                :    10 (  0.8%)

PRIORITY IDENTIFIERS FOR METABOLOMICS
  RefMet (Metabolomics Workbench)    :   279 ( 22.9%)
  CHEBI (Chemical Entities)          :   250 ( 20.5%)
  PubChem Compound                   :   153 ( 12.5%)
  HMDB (Human Metabolome)            :     0 (  0.0%)


### Quality Assessment

Identify patterns in unresolved or low-confidence mappings to improve future workflows.

In [15]:
# Pattern analysis for unresolved compounds
print("=" * 60)
print("UNRESOLVED COMPOUND PATTERNS")
print("=" * 60)

if unresolved:
    # Check for common patterns in names
    patterns = {
        "vendor_codes": [],     # Z-codes, AKOS, etc.
        "cas_numbers": [],       # CAS format: digits-digits-digit
        "generic_names": [],     # Compound names
    }
    
    for r in unresolved:
        name = r["query_name"]
        if re.match(r'^[A-Z]\d+', name):  # Pattern like Z12345
            patterns["vendor_codes"].append(name)
        elif re.match(r'^\d+-\d+-\d$', name):  # CAS number pattern
            patterns["cas_numbers"].append(name)
        else:
            patterns["generic_names"].append(name)
    
    print(f"  Vendor codes (Z-codes, AKOS, etc.): {len(patterns['vendor_codes'])}")
    print(f"  CAS numbers: {len(patterns['cas_numbers'])}")
    print(f"  Generic compound names: {len(patterns['generic_names'])}")
    
    if patterns["vendor_codes"]:
        print(f"\n  Sample vendor codes: {patterns['vendor_codes'][:5]}")
else:
    print("  No unresolved compounds!")

# Error analysis
print()
print("=" * 60)
print("ERROR ANALYSIS")
print("=" * 60)

if errors:
    error_types = {}
    for r in errors:
        err = str(r['error'])[:50]
        error_types[err] = error_types.get(err, 0) + 1
    
    print(f"Features with errors: {len(errors)}")
    print("\nError types:")
    for err, count in sorted(error_types.items(), key=lambda x: -x[1])[:5]:
        print(f"  {err}: {count}")
else:
    print("  No errors encountered!")

UNRESOLVED COMPOUND PATTERNS
  Vendor codes (Z-codes, AKOS, etc.): 19
  CAS numbers: 4
  Generic compound names: 24

  Sample vendor codes: ['Z1005800534', 'Z1637981622', 'Z2234928692', 'Z3408958828', 'Z2088772283']

ERROR ANALYSIS
  No errors encountered!


---
## Part 5: Export & Next Steps

### Export Results

Export results in two formats:
- **JSON**: Full detail, suitable for programmatic access
- **TSV**: Flat format, suitable for spreadsheet review

In [16]:
# Join API results back to original features if needed
name_to_result = {r["query_name"]: r for r in api_results}

full_results = []
for rec in mapping_records:
    api_result = name_to_result.get(rec["matched_name"], {})
    
    full_results.append({
        "feature_id": rec["feature_id"],
        "matched_name": rec["matched_name"],
        "match_level": rec.get("match_level"),
        "hmdb_hint": rec.get("hmdb_hint"),
        "resolved": api_result.get("resolved", False),
        "biomapper_curie": api_result.get("biomapper_curie"),
        "biomapper_name": api_result.get("biomapper_name"),
        "confidence_score": api_result.get("confidence_score"),
        "identifiers": api_result.get("identifiers", {}),
        "error": api_result.get("error"),
    })

# Calculate summary statistics
summary = {
    "total_features": len(df),
    "features_with_matched_name": len(mapping_records),
    "unique_names_queried": len(api_results),
    "resolved": len(resolved),
    "resolution_rate": len(resolved) / len(api_results) if api_results else 0,
    "vocabulary_coverage": vocab_counts,
}

# JSON export (full detail)
output_data = {
    "summary": summary,
    "mappings": full_results,
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(output_data, f, indent=2, default=str)
print(f"✓ Saved JSON: {OUTPUT_JSON}")

# TSV export (flat format for spreadsheet review)
flat_results = []
for r in full_results:
    # Flatten identifiers for TSV
    hmdb_ids = ";".join(r["identifiers"].get("HMDB", []))
    pubchem_ids = ";".join(r["identifiers"].get("PUBCHEM.COMPOUND", r["identifiers"].get("PUBCHEM", [])))
    chebi_ids = ";".join(r["identifiers"].get("CHEBI", []))
    refmet_ids = ";".join(r["identifiers"].get("refmet_id", []))
    
    flat = {
        "feature_id": r["feature_id"],
        "matched_name": r["matched_name"],
        "match_level": r["match_level"] or "",
        "hmdb_hint": r["hmdb_hint"] or "",
        "resolved": r["resolved"],
        "biomapper_curie": r["biomapper_curie"] or "",
        "confidence_score": r["confidence_score"] or "",
        "hmdb_ids": hmdb_ids,
        "pubchem_ids": pubchem_ids,
        "chebi_ids": chebi_ids,
        "refmet_ids": refmet_ids,
        "error": r["error"] or "",
    }
    flat_results.append(flat)

results_df = pd.DataFrame(flat_results)
results_df.to_csv(OUTPUT_TSV, sep="\t", index=False)
print(f"✓ Saved TSV: {OUTPUT_TSV}")

# Preview output
print()
print("=" * 60)
print("SAMPLE OUTPUT")
print("=" * 60)
results_df.head(5)

✓ Saved JSON: /home/trentleslie/Insync/projects/biovector-eval/data/metabolon/processed/tutorial_biomapper_mapping.json
✓ Saved TSV: /home/trentleslie/Insync/projects/biovector-eval/data/review/tutorial_biomapper_mapping.tsv

SAMPLE OUTPUT


,feature_id,matched_name,match_level,hmdb_hint,resolved,biomapper_curie,confidence_score,hmdb_ids,pubchem_ids,chebi_ids,refmet_ids,error
0,method_1_29905,"1,3-Diphenylguanidine",MS2,,True,RM:0199503,2.474219,,,144319,RM0199503,
1,method_1_1434,"3-Amino-1,2,4-triazol",MS2,,True,MESH:C575581,2.418158,,,,,
2,method_1_17931,"4,6-DIOXOHEPTANOIC ACID",MS2,HMDB03349,True,HMDB:HMDB0003349,,,,,,
3,method_1_42235,"(2S,3S)-6'-methyl-3-phenylspiro[oxirane-2,7'-q...",MS2,,True,CHEBI:221577,2.415053,,,221577,,
4,method_1_25401,(N(1) + N(8))-acetylspermidine,CURATION,,True,EFO:0800091,1.613289,,,,RM0135724,


### Summary & Next Steps

## What You Learned

1. **API Configuration**: How to authenticate and connect to Biomapper2
2. **Entity Types**: Using `biolink:SmallMolecule` routes to metabolomics-specific annotators
3. **Identifier Hints**: HMDB hints improve resolution from 93.6% to 100%
4. **Preprocessing**: Strip quotes and `_CE` suffixes for better resolution
5. **Result Interpretation**: Understanding CURIEs, confidence scores, and assigned_ids

## Recommended Next Steps

1. **Review low-confidence mappings** (score < 1.0) for potential errors
2. **Cross-validate with HMDB hints** where original hints disagreed with results
3. **Investigate unresolved compounds** - most are vendor codes or knowledge gaps
4. **Use the TSV output** for manual curation in a spreadsheet

## Related Resources

- **Demographics mapping example**: `notebooks/demographics/demographics_to_biomapper2.ipynb`
- **Full Metabolon analysis**: `notebooks/metabolon/metabolon_to_biomapper.ipynb`
- **Results summary**: `data/review/annotation_improvement_summary.md`

---
## Appendix: Edge Cases and Troubleshooting

### Common Unresolved Compound Patterns

When compounds fail to resolve, check for these patterns:

| Pattern | Example | Resolution |
|---------|---------|------------|
| **Vendor codes** | `Z12345`, `AKOS123` | These are internal identifiers - try alternative name sources |
| **CAS numbers** | `123-45-6` | CAS numbers alone rarely resolve - need compound name |
| **Stereochemistry issues** | `(R)-carnitine` vs `L-carnitine` | May resolve to different entities - verify manually |
| **Truncated names** | `3-Amino-1,2,4-triaz...` | Check original data for full name |

### Troubleshooting API Issues

| Issue | Solution |
|-------|----------|
| `401 Unauthorized` | Check API key is correctly set in environment |
| `429 Too Many Requests` | Increase `RATE_LIMIT_DELAY` (try 0.5s or 1.0s) |
| `500 Internal Server Error` | Report to API maintainers with example payload |
| Empty results | Verify entity_type matches your data (SmallMolecule for metabolites) |

### Tips for Large Datasets

1. **Start small**: Test with `LIMIT=50` before processing thousands
2. **Save progress**: Checkpoint results every N compounds for long runs
3. **Handle interruptions**: Load saved results and skip already-processed compounds
4. **Batch by category**: Process high-priority compounds first